# 组合回测（MultiSystem）

自 2.8.x 起，原组合回测组件 `Portfolio` / `AllocateFunds` 已移除，统一由 `MultiSystem` 承接：将多个 `System` 实例（单证券或嵌套的 `MultiSystem`）聚合为一个组合，父账户统一记账与下单。

支持两种运行模式：

- **模式 A（信号汇总，默认）**：父给每个子系统固定「影子账户」，父按权重（默认等权）统一分配并下单，子系统为纯信号源；
- **模式 B（资金划拨 / FOF-MOM）**：父给每个子系统「由上层分配额度的真实账户」，子系统在自己额度内自主决策，L2 透传。

**注意**：子系统必须各自持有独立的 SG/MM 实例（不同证券需各自计算信号，共享同一 SG 会被互相覆盖）。

In [ ]:
from hikyuu.interactive import *

## 创建子系统

从沪深300成分股中选取前 3 只，每只股票创建一个独立的 `SYS_Simple`（各自独立 SG/MM，并预运行绑定 K 线）。

In [ ]:
q = Query(-500)
# 从沪深300板块选取有 K 线数据的股票（不足时回退到其他候选）
stocks = [s for s in sm.get_block("指数板块", "沪深300") if s.valid and len(s.get_kdata(q)) > 10][:3]
if len(stocks) < 2:
    stocks = [s for c in ["sh000001", "sh000002", "sh000003", "sz000001", "sz000002"]
              for s in [sm[c]] if not s.is_null() and len(s.get_kdata(q)) > 10][:3]
print("selected:", [s.market_code for s in stocks])

def make_sys(stk):
    sg = SG_Flex(EMA(CLOSE(), n=5), slow_n=10)   # 每个子系统独立的信号
    mm = MM_FixedCount(100)                       # 每个子系统独立的资金管理
    sys = SYS_Simple(tm=crtTM(init_cash=100000), sg=sg, mm=mm)
    sys.run(stk.get_kdata(q))                     # 预运行以绑定 K 线（子系统各自的 TO）
    return sys

sys_list = [make_sys(s) for s in stocks]

## 模式 A：聚合回测

创建 `MultiSystem`，指定父账户（200 万初始资金）与调仓周期（每 10 个交易日再平衡），添加子系统后以沪深300指数（`sh000300`）为对齐时间轴驱动。

In [ ]:
my_tm = crtTM(Datetime(200101010000), 2000000)
ms = MultiSystem()
ms.tm = my_tm
ms.set_adjust_cycle(10)          # 调仓周期：每 10 个收盘日再平衡
for s in sys_list:
    ms.add(s)

# 以沪深300指数（sh000300）为对齐时间轴驱动（无该指数数据时回退到首个子系统的标的）
driver = sm['sh000300']
driver = driver if not driver.is_null() and len(driver.get_kdata(q)) > 10 else stocks[0]
ms.run(driver.get_kdata(q))

In [ ]:
from collections import Counter
trades = my_tm.get_trade_list()
cnt = Counter(t.stock.market_code for t in trades if t.business != BUSINESS.INIT)
print("trade count:", len(trades))
print("per stock:", dict(cnt))
print("total assets:", my_tm.get_funds().total_assets)
print("adjust turnover:", [(str(d), round(t, 4)) for d, t in ms.get_adjust_turnover()][:5])

## 模式 B：资金划拨（FOF-MOM）

切换到模式 B 后，父按权重（默认等权）将真实额度分配给各子系统（`set_sub_init_cash` 设置初始额度），子系统在自己额度内自主交易，父透传其指令；调仓日父回写下期额度。

In [ ]:
tm_b = crtTM(Datetime(200101010000), 2000000)
ms_b = MultiSystem()
ms_b.tm = tm_b
ms_b.set_mode("B")                    # 资金划拨模式
ms_b.set_sub_init_cash(500000)         # 子系统初始额度
ms_b.set_adjust_cycle(10)
for s in sys_list:
    ms_b.add(s)
ms_b.run(driver.get_kdata(q))

trades_b = tm_b.get_trade_list()
cnt_b = Counter(t.stock.market_code for t in trades_b if t.business != BUSINESS.INIT)
print("mode B trade count:", len(trades_b), "per stock:", dict(cnt_b))
print("mode B total assets:", tm_b.get_funds().total_assets)

## 绩效展示

父账户的绩效统计与绘图（与单证券一致，使用 `TradeManager` 接口）。

In [ ]:
per = my_tm.get_performance()
Performance_to_df(per)

In [ ]:
my_tm.performance(q, ref_stk=driver)